# 🌍 Territory Optimisation Engine v2

Reviews account ownership region by region and proposes **which accounts to move out of or into each AE's territory**, while respecting your business rules, and explains every move in plain English.

## ⚡ Quick start (5 minutes)
1. **Run cell 0** (installs the solver) and **cell 1** (the engine). Both are one-time per session.
2. **Run cell 2 (Settings).** Leave *use_demo_data* ticked to try it, or untick it and point it at your Excel file.
3. **Run cell 3.** You'll see progress per region, then a dashboard: headline KPIs, a plain-English summary, rule scorecard, workload chart, a flow diagram of the moves, and the list of moves with reasons.
4. **Run cell 4** to download the Excel workbook (also contains a *Moves* sheet for managers to approve/reject).
5. **Next cycle:** after managers have filled in the *Manager Decision* column, run **cell 5** to turn the reviewed workbook into the input file for your next run. Rejected moves become *locked* automatically.

## 🆕 What changed vs. the original script
| Area | Original | Now |
|---|---|---|
| Rules | All hard: one clash = "Failed" and nothing is exported for that region | **Soft mode (default)**: always returns the best compromise and tells you exactly which rules were bent and by how much. **Hard mode** keeps the original strict behaviour |
| Diagnostics | Three arithmetic guesses, wiped from the screen at the end | **Exact conflict detection** (the solver names the rules that clash) + a pre-flight check, both kept on screen and in the Excel file |
| Failed region | Accounts silently disappear from the export | Accounts **keep their current owner** and the region is flagged |
| Unassigned accounts | Not reported | Own table with a reason for each |
| Objective | Revenue only (all assignments equal, so only drops the smallest accounts) + hard-coded 5,000,000 retention bonus | Clear layers: **coverage > rules > stability > balance**, each tunable. $0 prospects are no longer dropped first |
| Moves | Not visible | Every move has *from → to* and a **plain-English reason** |
| Churn control | None | Max % of accounts moving, max in / max out per AE |
| Locked accounts | None | Optional `Locked` column: accounts that must not move (open deals, exec sponsors) |
| Owners who left | Not handled | Detected, and their accounts are reassigned and labelled |
| Speed | Variables for ineligible pairs created then ignored | Ineligible AE/account pairs removed before solving; pre-computed index lists; 8-worker portfolio; fixed seed (same input = same output) |
| Output | HTML + one chart | KPI dashboard, scorecard, workload-vs-band chart, Sankey of moves, 13-sheet Excel, scenario comparison |
| Data checks | None | Missing columns, duplicate IDs/AE names, unknown roles, bad numbers, regions without AEs, etc. |

> ✅ All original features and parameter names are kept. The growth-engine rule still caps an AE's **total** effort by default (`growth_cap_basis = "total"`); switch to `"growth_only"` if the intent is to cap only the effort on growth-engine accounts.

In [ ]:
#@title 📦 0. Install (run once per session) { display-mode: "form" }
%pip -q install "territory-optimisation-engine[notebook] @ git+https://github.com/5d7/territory-optimisation-engine.git@engine-v0.1.0"


In [ ]:
#@title 🧠 1. Engine – import the tested package { display-mode: "form" }
# TERRITORY OPTIMISATION ENGINE v2
from territory_optimisation import *
print('Engine loaded. Next: run the Settings cell.')


## 📂 Your input file

An Excel workbook with two sheets. Column names must match exactly.

**Sheet `Accounts`**
| Column | Required | Notes |
|---|---|---|
| Region | ✅ | Accounts are only assigned to AEs aligned to the same region |
| Product Focus | ✅ | `Enterprise AI` triggers the AI-specialist rule |
| Current AE | ✅ | Blank = new account. A name not found in `AE_Roles` = owner left, account is reassigned |
| Annual Revenue | ✅ | Number |
| Estimated Effort (Hours) | ✅ | Number (decimals are rounded) |
| Status | ✅ | `Customer` / `Prospect` |
| Tier | ✅ | `Tier 1` / `Tier 2` / `Tier 3` |
| Spend Map | ✅ | `Burn Risk`, `Growth Engine`, anything else is neutral |
| Vertical | ✅ | |
| Account ID | optional | Generated if missing. **Recommended**: keep it stable between cycles |
| Account Name | optional | Shown in the move lists |
| Locked | optional | `Yes` = this account must stay with its current AE |

**Sheet `AE_Roles`**
| Column | Notes |
|---|---|
| AE Name | Must be unique |
| Role | `Strategic`, `Account Manager`, `Enterprise Acquisition` |
| Region Alignment | |
| Is_AI_Specialist | TRUE/FALSE (Yes/No also fine) |

💡 Run the last cell, *Download a blank input template*, to get an example file.

## ⚙️ Settings guide (cell 2)

**Soft vs hard mode** – the most important new setting.
* **soft** (recommended): rules are strongly *protected* but can be bent when there is no other way. You always get a plan, and the *Rule Scorecard* shows what is still outside the rules and by how much.
* **hard**: the original behaviour – every rule must hold or the region fails. If it fails, the tool tells you *which rules* clash.

**Stability & churn**
* `stability_bonus` – reward for leaving an account with its current owner. **Higher = fewer moves.** At 0 the optimiser moves anything that improves the score.
* `max_moves_pct` – at most this % of currently-owned accounts may change owner in one cycle (100 = no limit).
* `max_in_per_ae` / `max_out_per_ae` – limit how many accounts any one AE can receive / lose (0 = no limit). Handy to avoid disrupting a rep's book.
* `require_full_coverage` – force every account to get an owner (otherwise a few may be left out when rules/capacity leave no room).
* `respect_locked` – honour the optional `Locked` column.

**Workload fairness** – `allowed_workload_variance_pct` defines the band around the regional average effort (target = total regional effort ÷ number of AEs). 20% on a 2,400h average means every AE should land between 1,920h and 2,880h.

**Advanced weights (section 8)** – only used in soft mode. They set the *priority order*; the defaults give: **assigning accounts > keeping rules > keeping accounts with their owner > perfect balance**. Raise a `w_…` value to protect that rule harder; lower it to make it more negotiable.

**Solver** – `max_time_per_region_seconds` is a ceiling; the solver stops earlier when it is within 0.5% of the best possible plan. `parallel_regions` solves regions at the same time (useful with many regions). `random_seed` makes runs repeatable.

In [ ]:
#@title ⚙️ 2. Settings – change the sliders, then run this cell { display-mode: "form" }
#@markdown ### 📂 Data source
use_demo_data = True #@param {type:"boolean"}
#@markdown Tick **use_demo_data** to try the tool with built-in sample data. Untick it to use your own Excel file (you will be asked to upload it).
input_file = "Enterprise_Scale_Accounts.xlsx" #@param {type:"string"}
accounts_sheet = "Accounts" #@param {type:"string"}
aes_sheet = "AE_Roles" #@param {type:"string"}

#@markdown ---
#@markdown ### 👔 1. Role limits
global_max_accounts = 150 #@param {type:"slider", min:50, max:200, step:10}
strategic_max_accounts = 80 #@param {type:"slider", min:20, max:150, step:10}
am_max_prospects = 40 #@param {type:"slider", min:10, max:100, step:5}

#@markdown ---
#@markdown ### 🌐 2. Geographic & product alignment
ai_specialists_only = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🧹 3. "Grunt work" & vertical anti-silo
min_tier_3_per_ae = 20 #@param {type:"slider", min:0, max:60, step:5}
max_accounts_per_vertical = 40 #@param {type:"slider", min:10, max:100, step:5}

#@markdown ---
#@markdown ### 💰 4. Spend Map portfolio balancing
max_burn_risk_per_ae = 25 #@param {type:"slider", min:5, max:50, step:5}
max_growth_engines = 25 #@param {type:"slider", min:5, max:50, step:5}
growth_engine_effort_cap = 3000 #@param {type:"slider", min:1000, max:5000, step:100}
growth_cap_basis = "total" #@param ["total", "growth_only"]

#@markdown ---
#@markdown ### ⚖️ 5. Fairness
allowed_workload_variance_pct = 20 #@param {type:"slider", min:5, max:50, step:5}

#@markdown ---
#@markdown ### 🔄 6. Stability & churn control (NEW)
constraint_mode = "soft" #@param ["soft", "hard"]
stability_bonus = 1000 #@param {type:"slider", min:0, max:5000, step:100}
max_moves_pct = 100 #@param {type:"slider", min:0, max:100, step:5}
max_in_per_ae = 0 #@param {type:"slider", min:0, max:60, step:1}
max_out_per_ae = 0 #@param {type:"slider", min:0, max:60, step:1}
require_full_coverage = False #@param {type:"boolean"}
respect_locked = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### ⏱️ 7. Solver
max_time_per_region_seconds = 120 #@param {type:"slider", min:30, max:600, step:30}
parallel_regions = False #@param {type:"boolean"}
random_seed = 42 #@param {type:"integer"}

#@markdown ---
#@markdown ### 🎛️ 8. Advanced – how strongly each rule is protected in soft mode (leave defaults unless tuning)
balance_weight = 5 #@param {type:"slider", min:0, max:50, step:1}
base_value_per_account = 3000 #@param {type:"slider", min:0, max:10000, step:500}
w_cap = 20000 #@param {type:"integer"}
w_tier3 = 5000 #@param {type:"integer"}
w_burn = 5000 #@param {type:"integer"}
w_strategic_mix = 8000 #@param {type:"integer"}
w_am_prospects = 8000 #@param {type:"integer"}
w_vertical = 5000 #@param {type:"integer"}
w_growth_cap = 50 #@param {type:"integer"}
w_effort_band = 50 #@param {type:"integer"}
w_moves = 8000 #@param {type:"integer"}
w_inout = 5000 #@param {type:"integer"}

cfg = Config(
    use_demo_data=use_demo_data, input_file=input_file, accounts_sheet=accounts_sheet, aes_sheet=aes_sheet,
    global_max_accounts=global_max_accounts, strategic_max_accounts=strategic_max_accounts, am_max_prospects=am_max_prospects,
    ai_specialists_only=ai_specialists_only, min_tier_3_per_ae=min_tier_3_per_ae, max_accounts_per_vertical=max_accounts_per_vertical,
    max_burn_risk_per_ae=max_burn_risk_per_ae, max_growth_engines=max_growth_engines,
    growth_engine_effort_cap=growth_engine_effort_cap, growth_cap_basis=growth_cap_basis,
    allowed_workload_variance_pct=allowed_workload_variance_pct, constraint_mode=constraint_mode,
    stability_bonus=stability_bonus, max_moves_pct=max_moves_pct, max_in_per_ae=max_in_per_ae, max_out_per_ae=max_out_per_ae,
    require_full_coverage=require_full_coverage, respect_locked=respect_locked,
    max_time_per_region_seconds=max_time_per_region_seconds, parallel_regions=parallel_regions, random_seed=random_seed,
    balance_weight=balance_weight, base_value_per_account=base_value_per_account,
    w_cap=w_cap, w_tier3=w_tier3, w_burn=w_burn, w_strategic_mix=w_strategic_mix, w_am_prospects=w_am_prospects,
    w_vertical=w_vertical, w_growth_cap=w_growth_cap, w_effort_band=w_effort_band, w_moves=w_moves, w_inout=w_inout).validate()
print(f"Settings saved. Mode: {cfg.constraint_mode.upper()} | data: {'DEMO' if cfg.use_demo_data else cfg.input_file} | "
      f"up to {cfg.max_time_per_region_seconds}s per region")

In [ ]:
#@title ▶️ 3. Run the territory review { display-mode: "form" }
result = run_territory_review(cfg)
show_dashboard(result)

## 📊 Reading the results

* **Headline cards** – coverage (accounts with an owner), stability (accounts that kept their owner), proposed moves, rule breaches *before → after*, pipeline covered, workload spread.
* **Region status** – `OPTIMAL` = best plan proven; `FEASIBLE` = good plan but more time might improve it (see *Plan quality (gap %)*; under ~2% is excellent); `INFEASIBLE` (hard mode) = rules clash, the note names them; `NO_AES` = region has accounts but no AEs.
* **Rule scorecard** – for every rule: how many AEs broke it before vs after and by how much in total. This is the table to share with leadership.
* **Workload chart** – bars are the proposed hours per AE, diamonds are today's hours, green dashes are the allowed band.
* **Moves** – every account that changes hands, with the reason (e.g. *"Relieves AE-05, who was outside the rules: Max Burn Risk…"*, *"Enterprise AI account needs an AI specialist"*).
* **Unassigned** – accounts nobody could take and why.

### Account `Action` values in the Excel
`Keep` · `Keep (locked)` · `Move` · `New assignment` (had no owner) · `Reassigned (owner unavailable)` · `Unassigned`

### Excel workbook sheets
README · Run Summary · Region Status · **Moves** (with Manager Decision dropdowns) · Rule Scorecard · AE Summary (before/after per rep) · Assignments (all accounts) · Unassigned · Rule Detail (every check) · Pre-flight · Data Quality · Settings · AE_Roles

In [ ]:
#@title 💾 4. Download the Excel workbook { display-mode: "form" }
export_excel(result)

## 🔁 The continuous review cycle

1. **Run** the review (cells 3–4) and send the workbook to sales managers.
2. Managers open the **Moves** sheet and pick **Approve** or **Reject** per account (blank = Approve). They can also put `Yes` in *Lock next cycle* to freeze an account.
3. Upload the reviewed workbook in **cell 5**. It produces `Next_Cycle_Input.xlsx` where:
   * approved moves become the new *Current AE*,
   * rejected moves stay with the current AE and are **locked**,
   * everything else is carried over.
   (Reassignments of accounts whose owner left can't be rejected - they must have an owner.)
4. Next cycle, set *use_demo_data* off, point *input_file* at `Next_Cycle_Input.xlsx` and run again. Changes in headcount, new accounts or revised ratings flow through automatically.

**Scenario testing (cell 6)** – compare, say, 10% vs 35% workload tolerance or "max 10% of accounts move" before committing to a policy.

In [ ]:
#@title 🔁 5. Next cycle: build the input file from the managers' reviewed workbook { display-mode: "form" }
#@markdown Leave blank to be asked to upload the reviewed workbook (the one you downloaded in step 4, with the *Manager Decision* column filled in).
reviewed_file = "" #@param {type:"string"}
if not reviewed_file:
    _up = colab_files.upload()
    reviewed_file = list(_up.keys())[0]
build_next_cycle_input(reviewed_file)

In [ ]:
#@title 🧪 6. (Optional) Compare scenarios side by side { display-mode: "form" }
#@markdown Each scenario is a quick re-run with some settings changed. Edit the dictionary below to try your own ideas.
scenario_seconds = 30 #@param {type:"slider", min:15, max:120, step:15}
scenarios = {
    "Tighter workload (10%)": dict(allowed_workload_variance_pct=10),
    "Looser workload (35%)":  dict(allowed_workload_variance_pct=35),
    "Max 10% of accounts move": dict(max_moves_pct=10),
    "Strict rules (hard mode)": dict(constraint_mode="hard"),
}
scenario_table = run_scenarios(cfg, scenarios, seconds=scenario_seconds)

## 🛠️ Troubleshooting

| Symptom | What to do |
|---|---|
| "Blocking data problems found" | Read the table printed above the error: missing columns, duplicate IDs or AE names. |
| Region `INFEASIBLE` (hard mode) | The note lists the clashing rules. Relax one of them, or switch to **soft** mode. |
| Region `FEASIBLE`, gap > 5% | Raise `max_time_per_region_seconds`, or accept: the plan is valid, just maybe improvable. |
| Many accounts unassigned | Check the *Pre-flight* warnings (capacity, Tier 3 shortage, vertical caps, no eligible AI specialist). Raise `global_max_accounts`, or tick `require_full_coverage` (soft mode will then bend rules instead of dropping accounts). |
| Too many moves | Raise `stability_bonus` and/or set `max_moves_pct`. |
| Rules still breached in soft mode | Raise that rule's `w_…` weight (section 8), or relax the rule itself. |
| Growth-engine rule behaves like a hard limit | If the workload floor is above `growth_engine_effort_cap` no AE can hold more than `max_growth_engines` growth accounts. The pre-flight warns you; raise the cap or use `growth_only`. |
| Run is slow | Lower the time limit, use `parallel_regions`, or runtime ▸ change to a high-RAM/CPU instance. |

## 🧳 Moving off Colab later
Everything Colab-specific is isolated in this notebook's *form* cells (`#@param` sliders), `colab_files` upload/download calls and the on-screen dashboard. The engine (cell 1) is plain Python split into: **Config → data → model → rule checks → orchestrator → reporting**. To migrate:
* **Scheduler / pipeline**: call `run_territory_review(Config(...), df_accounts, df_aes)` and write `result.accounts`, `result.moves`, `result.ae_summary`, `result.rule_detail`, `result.scorecard`, `result.region_summary` to your warehouse. These are tidy DataFrames ready for Power BI / Tableau / Looker.
* **Web app for non-technical users**: put the same calls behind Streamlit or Dash and map the settings sliders to `Config` fields.
* **Different data source**: replace `load_data()` only; nothing else reads Excel.

In [ ]:
#@title 📄 (Optional) Download a blank input template { display-mode: "form" }
download_input_template()